In [0]:
import time
from pyspark.sql import functions as F

LANDING = "/Volumes/nyc_taxi/bronze/streaming_landing"

schema = spark.table("nyc_taxi.silver.trips_clean").schema
 
stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/nyc_taxi/bronze/raw_files/_schema/stream")
    .schema(schema)
    .load(LANDING))
 
stream = stream.withColumn("_stream_ingested_at", F.current_timestamp())
 
query = (stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/nyc_taxi/bronze/raw_files/_checkpoints/trips_stream")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("nyc_taxi.bronze.trips_streaming"))
 
query.awaitTermination()
